# Project Success Classification
This notebook predicts whether a project succeeds (`success_flag`) and compares multiple algorithms.

Workflow: **compréhension des données** → **nettoyage explicite (dans ce notebook)** → **pré-traitement sklearn (imputation / encodage / scaling dans un pipeline ajusté sur le train)** → entraînement → métriques → graphiques.

In [ ]:
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, roc_curve, ConfusionMatrixDisplay

ROOT = Path.cwd().resolve().parents[0] if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if (ROOT / 'ml-workspace').exists():
    ROOT = ROOT / 'ml-workspace'
DATA_PATH = ROOT / 'data/raw/modeling_dataset.csv'
RESULTS_PLOTS = ROOT / 'results/plots'
RESULTS_METRICS = ROOT / 'results/metrics'
RESULTS_PLOTS.mkdir(parents=True, exist_ok=True)
RESULTS_METRICS.mkdir(parents=True, exist_ok=True)
SEED = 42

In [ ]:
df = pd.read_csv(DATA_PATH)
display(df.head())
print(df.info())
print(df['success_flag'].value_counts(normalize=True).rename('class_ratio'))

## Exploration et diagnostic
Contrôle des valeurs manquantes et d'un résumé statistique avant nettoyage.

In [ ]:
null_report = df.isna().mean().sort_values(ascending=False)
display(null_report[null_report > 0])
display(df.describe(include='all').T.head(20))

## Nettoyage des données (étapes explicites)
1. Suppression des doublons complets.
2. Suppression des lignes sans cible.
3. Imputation **dans le notebook** des numériques (médiane) et des catégorielles (mode) pour documenter le traitement ; le `Pipeline` garde les mêmes étapes ajustées **uniquement sur le train** pour éviter la fuite d'information au moment du `fit`.

In [ ]:
target = 'success_flag'
clean_df = df.copy()
n0 = len(clean_df)
clean_df = clean_df.drop_duplicates().reset_index(drop=True)
print(f'Doublons supprimés: {n0 - len(clean_df)} ligne(s)')
clean_df = clean_df.dropna(subset=[target]).copy()
print('Lignes après suppression des NaN sur la cible:', len(clean_df))

null_before = clean_df.isna().sum()
num_cols_all = clean_df.select_dtypes(include=[np.number]).columns.tolist()
for c in num_cols_all:
    if clean_df[c].isna().any():
        med = clean_df[c].median()
        nmiss = clean_df[c].isna().sum()
        clean_df[c] = clean_df[c].fillna(med)
        print(f'Imputation médiane {c}: {nmiss} valeur(s) -> {med:.4g}')
cat_cols_all = clean_df.select_dtypes(exclude=[np.number]).columns.tolist()
if target in cat_cols_all:
    cat_cols_all.remove(target)
for c in cat_cols_all:
    if clean_df[c].isna().any():
        mode = clean_df[c].mode(dropna=True)
        fill = mode.iloc[0] if len(mode) else 'unknown'
        nmiss = clean_df[c].isna().sum()
        clean_df[c] = clean_df[c].fillna(fill)
        print(f'Imputation mode {c}: {nmiss} valeur(s) -> {fill}')

null_after = clean_df.isna().sum()
display(pd.DataFrame({'avant': null_before, 'apres': null_after}).fillna(0).astype(int).T)

## Pré-traitement (train / test + `ColumnTransformer`)
Les transformateurs ci-dessous sont **fit** uniquement sur `X_train` via le `Pipeline` pour respecter le protocole anti-fuite.

In [ ]:
drop_cols = ['project_id','title','status','client_satisfaction_score', target]
features = [c for c in clean_df.columns if c not in drop_cols]
X, y = clean_df[features], clean_df[target]

num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = X.select_dtypes(exclude=[np.number]).columns.tolist()

preprocessor = ColumnTransformer([
    ('num', Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('standardize', StandardScaler())
    ]), num_cols),
    ('cat', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encode', OneHotEncoder(handle_unknown='ignore'))
    ]), cat_cols)
])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.22, random_state=SEED, stratify=y)
print('Train shape:', X_train.shape, 'Test shape:', X_test.shape)

In [ ]:
models = {
    'logistic_regression': LogisticRegression(max_iter=400, random_state=SEED),
    'random_forest_classifier': RandomForestClassifier(n_estimators=240, random_state=SEED),
    'gradient_boosting_classifier': GradientBoostingClassifier(random_state=SEED)
}

rows = []
roc_store = []
for name, model in models.items():
    pipe = Pipeline([('prep', preprocessor), ('model', model)])
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    if hasattr(pipe.named_steps['model'], 'predict_proba'):
        y_score = pipe.predict_proba(X_test)[:, 1]
    else:
        raw = pipe.decision_function(X_test)
        y_score = (raw - raw.min()) / (raw.max() - raw.min() + 1e-9)

    rows.append({
        'algorithm': name,
        'accuracy': accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred, zero_division=0),
        'recall': recall_score(y_test, y_pred, zero_division=0),
        'f1': f1_score(y_test, y_pred, zero_division=0),
        'roc_auc': roc_auc_score(y_test, y_score)
    })

    ConfusionMatrixDisplay.from_predictions(y_test, y_pred, cmap='Blues')
    plt.title(f'Confusion Matrix - {name}')
    plt.tight_layout()
    plt.savefig(RESULTS_PLOTS / f'classification_confusion_{name}.png', dpi=160)
    plt.show()

    fpr, tpr, _ = roc_curve(y_test, y_score)
    roc_store.append((name, fpr, tpr))

metrics_df = pd.DataFrame(rows).sort_values('f1', ascending=False)
display(metrics_df)
metrics_df.to_csv(RESULTS_METRICS / 'classification_metrics.csv', index=False)
metrics_df.to_json(RESULTS_METRICS / 'classification_metrics.json', orient='records', indent=2)

In [ ]:
plt.figure(figsize=(7,5))
for name, fpr, tpr in roc_store:
    plt.plot(fpr, tpr, label=name)
plt.plot([0,1],[0,1],'k--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves - All Classification Algorithms')
plt.legend()
plt.tight_layout()
plt.savefig(RESULTS_PLOTS / 'classification_roc_all_algorithms.png', dpi=160)
plt.show()
print('Best model based on F1:', metrics_df.iloc[0]['algorithm'])